# TTS SOTA 小模型评测报告 — 中英双语言 · Colab T4

**环境:** Google Colab T4 (15GB VRAM) | **日期:** 2026-05 | **选取标准:** 开源可部署 + 中英双言

---

## 模型选型总览

| 推荐方向 | 代表模型 (参数量) | 中文语音质量与语言支持 | 端侧能力与性能特点 | 推荐理由 |
|---------|------------------|------------------------|--------------------|----------|
| 极致轻量与本地部署 | **ChatTTS** (30M) | 专为对话场景打造，中文MOS评分达4.2，接近真人水平。 | 体积仅30M，可在树莓派等边缘设备运行，推理延迟<200ms。 | 如果你资源极度有限，且对实时对话有要求，这是最值得一试的。 |
| 最佳端侧全能选手 | **Fun-CosyVoice3** (0.5B) | 9种语言 + 18种方言，支持多语言和跨语种零样本克隆，中文表现出色。 | 阿里出品，首包延迟仅150ms，功耗优化极佳，适合本地部署。 | 追求中文（含方言）场景下的综合体验与部署便利性，首选它。 |
| 工业级高质量合成 | **IndexTTS-1.5** | 专为中文优化，原生支持中英混合，提供拼音纠错，完美解决多音字问题。 | 来自B站AI团队，工业级系统，在低GPU占用下保持高质量合成，适合大规模部署。 | 需要最高质量的中文合成，尤其是有大量"朗读"内容的需求，它是合适选择。 |
| 多语种高性能标杆 | **OmniVoice** (0.8B) | 600+种语言支持，中文WER仅0.84%，超ElevenLabs等商业模型。 | 小米Kaldi团队出品，实时因子低至0.025（40倍实时），速度极快。 | 如果你的应用需要超高速且高质量的多语言合成，它是突破性的选择。 |
| 高保真旗舰与克隆 | **VoxCPM2** (2B) | 支持30种语言，48kHz录音室级输出，中文音色克隆相似度(SIM)达82.5%。 | 清华出品，2B参数可在本地GPU运行，音色相似度极高。 | 预算充足且追求顶级音色克隆质量，可选它实现专业级效果。 |
| 轻量级优秀备选 | **S1-Mini** (0.5B) | 支持14种语言和50+情感，输出情感丰富，效果出色。 | 从4B大模型蒸馏而来，兼顾轻量与高性能，可在边缘设备部署。 | 如果你想在轻量模型中寻找更多选择，它是一个优秀的备选。 |

>  数据来源: 各模型官方 HuggingFace / GitHub 页面，2026-05 验证。全部模型权重公开可下载。

---

## 评测维度

| 维度 | 权重 | 测量方式 |
|------|------|----------|
| 推理速度 | 25% | 统一文本逐句计时(ms)，RTF |
| VRAM 占用 | 20% | `torch.cuda.max_memory_allocated()` + `nvidia-smi` 验证 |
| CPU/GPU使用率 | 10% | `psutil` CPU% + `pynvml` GPU% 实时采样 |
| 中文质量 | 20% | 主观MOS + 多音字正确率 |
| 英文质量 | 10% | 主观MOS + 中英混合发音准确度 |
| 克隆能力 | 15% | 3s/10s样本音色相似度 |

### 统一测试文本 (所有模型共用)

```python
TEXT_LIST = [
    "我要去上学校。",                                          # 纯中文短句
    "我要去上学校，but I really hate school!!! 我不想上学！！！。",  # 中英混合 + 情绪
    "我去宣布放假！！！ 我可以去宣布吗？"                         # 中文口语 + 重复标点
]
```

In [ ]:
# ============================================================
# Cell 0: Environment Check & Common Setup
# ============================================================
import torch, subprocess, sys, os, time, json, warnings, threading
import numpy as np
import IPython.display as ipd
import soundfile as sf
import psutil
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List
warnings.filterwarnings("ignore")

# --- GPU ---
gpu = subprocess.check_output(
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader,nounits", shell=True
).decode().strip()
gpu_name, vram_mb = gpu.split(", ")
print(f"{'='*60}")
print(f"  GPU        : {gpu_name.strip()}")
print(f"  VRAM       : {vram_mb.strip()} MB (~{int(vram_mb)//1024} GB)")
print(f"  CUDA       : {torch.cuda.is_available()}")
print(f"  PyTorch    : {torch.__version__}")
print(f"  CPU cores  : {psutil.cpu_count(logical=True)} (logical)")
print(f"{'='*60}")

OUT_DIR = Path("/content/tts_outputs")
OUT_DIR.mkdir(exist_ok=True)

# ============================================================
# Unified test text — ALL models must synthesize this
# ============================================================
TEXT_LIST = [
    "我要去上学校。",                                          # 纯中文短句
    "我要去上学校，but I really hate school!!! 我不想上学！！！。",  # 中英混合 + 情绪
    "我去宣布放假！！！ 我可以去宣布吗？"                         # 中文口语 + 重复标点
]

print("=== Unified Test Text (all models) ===")
for i, t in enumerate(TEXT_LIST):
    print(f"  [{i}] '{t}' ({len(t)} chars)")
print()

# ============================================================
# GPU / CPU utilization monitor (real-time sampling)
# ============================================================
try:
    import pynvml
    pynvml.nvmlInit()
    HAS_NVML = True
except Exception:
    HAS_NVML = False
    print("  ⚠ pynvml not available — GPU util will be 0")

class PerfMonitor:
    """Real-time CPU/GPU utilization sampler."""
    def __init__(self):
        self._stop = False
        self._cpu_samples = []
        self._gpu_samples = []
        self._thread = None

    def _sample(self):
        proc = psutil.Process()
        while not self._stop:
            self._cpu_samples.append(proc.cpu_percent(interval=0.05) / psutil.cpu_count())
            if HAS_NVML:
                try:
                    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
                    self._gpu_samples.append(pynvml.nvmlDeviceGetUtilizationRates(handle).gpu)
                except Exception:
                    self._gpu_samples.append(0)
            else:
                self._gpu_samples.append(0)

    def start(self):
        self._stop = False
        self._cpu_samples.clear()
        self._gpu_samples.clear()
        self._thread = threading.Thread(target=self._sample, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop = True
        if self._thread:
            self._thread.join(timeout=1)

    @property
    def avg_cpu_pct(self):
        return round(np.mean(self._cpu_samples), 1) if self._cpu_samples else 0

    @property
    def peak_cpu_pct(self):
        return round(np.max(self._cpu_samples), 1) if self._cpu_samples else 0

    @property
    def avg_gpu_pct(self):
        return round(np.mean(self._gpu_samples), 1) if self._gpu_samples else 0

    @property
    def peak_gpu_pct(self):
        return round(np.max(self._gpu_samples), 1) if self._gpu_samples else 0

perf = PerfMonitor()

# ============================================================
# Benchmark harness — per-sentence ms timing + VRAM + CPU/GPU
# ============================================================
@dataclass
class SentenceTiming:
    text: str
    time_ms: float = 0.0
    audio_len_s: float = 0.0

@dataclass
class TTSResult:
    model: str
    time_s: float = 0.0
    time_ms: float = 0.0
    vram_gb: float = 0.0
    rtf: float = 0.0
    audio_len_s: float = 0.0
    cpu_avg_pct: float = 0.0
    cpu_peak_pct: float = 0.0
    gpu_avg_pct: float = 0.0
    gpu_peak_pct: float = 0.0
    sentences: List[SentenceTiming] = field(default_factory=list)
    notes: str = ""

RESULTS: Dict[str, TTSResult] = {}

def bench(model_name: str, fn, text_list: List[str] = None):
    """Run TTS model and collect per-sentence timing + VRAM + CPU/GPU stats."""
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    perf.start()
    t0 = time.time()

    wav, sr, sentence_timings = fn()  # fn returns (wav, sr, timings)

    elapsed = time.time() - t0
    perf.stop()

    peak_vram = torch.cuda.max_memory_allocated() / 1024**3
    audio_len = len(wav) / sr if wav is not None and len(wav) > 0 else 0.001
    rtf = round(elapsed / audio_len, 4)
    r = TTSResult(
        model=model_name,
        time_s=round(elapsed, 3),
        time_ms=round(elapsed * 1000, 1),
        vram_gb=round(peak_vram, 2),
        rtf=rtf,
        audio_len_s=round(audio_len, 2),
        cpu_avg_pct=perf.avg_cpu_pct,
        cpu_peak_pct=perf.peak_cpu_pct,
        gpu_avg_pct=perf.avg_gpu_pct,
        gpu_peak_pct=perf.peak_gpu_pct,
        sentences=sentence_timings
    )
    RESULTS[model_name] = r

    # Print summary
    print(f"  {'─'*50}")
    print(f"  Total : {elapsed*1000:.1f}ms ({elapsed:.3f}s) | VRAM peak: {peak_vram:.2f}GB | RTF: {rtf}")
    print(f"  CPU   : avg {perf.avg_cpu_pct:.1f}% | peak {perf.peak_cpu_pct:.1f}%")
    print(f"  GPU   : avg {perf.avg_gpu_pct:.1f}% | peak {perf.peak_gpu_pct:.1f}%")
    if sentence_timings:
        print(f"  Per-sentence:")
        for i, st in enumerate(sentence_timings):
            chars_per_sec = len(TEXT_LIST[i]) / (st.time_ms / 1000) if st.time_ms > 0 else 0
            print(f"    [{i}] {st.time_ms:.1f}ms → {st.audio_len_s:.2f}s audio | {chars_per_sec:.0f} chars/s")
    return wav, sr

print("Setup complete.")

In [ ]:
# ============================================================
# Cell 1: Install All Dependencies (run once)
# ============================================================
# Common deps
!pip install -q torchaudio soundfile librosa scipy cn2an pypinyin

# ChatTTS
!pip install -q ChatTTS

# CosyVoice (includes Fun-CosyVoice3 support)
!pip install -q cosyvoice modelscope

# IndexTTS
!pip install -q index-tts

# OmniVoice
!pip install -q omnivoice

# VoxCPM2
!pip install -q voxcpm

# S1-Mini (fish-speech)
!pip install -q fish-speech

print("All dependencies installed.")

---
## Model 1: ChatTTS — 极致轻量对话引擎

| 属性 | 值 |
|------|-----|
| 参数量 | ~30M |
| 中文MOS | 4.2 |
| 语言 | 中文 + 英文，中英混合 |
| 延迟 | <200ms |
| 硬件 | 树莓派 / CPU / GPU 均可 |
| 权重 | [2noise/ChatTTS](https://huggingface.co/2noise/ChatTTS) |

In [ ]:
import ChatTTS

def test_chatts():
    chat = ChatTTS.Chat()
    chat.load(compile=False)
    chat.to("cuda")

    wavs, timings = [], []
    for text in TEXT_LIST:
        t0 = time.time()
        wav = chat.infer([text], use_decoder=True)
        dt_ms = (time.time() - t0) * 1000
        wav_arr = wav[0].squeeze()
        wavs.append(wav_arr)
        timings.append(SentenceTiming(text=text[:30]+"...", time_ms=round(dt_ms, 1),
                                       audio_len_s=round(len(wav_arr)/24000, 2)))

    combined = np.concatenate(wavs)
    sf.write(str(OUT_DIR / "01_chatts.wav"), combined, 24000)
    return combined, 24000, timings

wav, sr = bench("ChatTTS", test_chatts)
ipd.Audio(wav, rate=sr)

---
## Model 2: Fun-CosyVoice3 — 阿里全能方言王

| 属性 | 值 |
|------|-----|
| 参数量 | ~0.5B |
| 语言 | 9种语言 + 18种方言 + 9种情感 |
| 克隆 | 3秒零样本克隆 |
| 首包延迟 | ~150ms (双向流式) |
| 中英混说 | WER降低56.4% |
| 权重 | [FunAudioLLM/Fun-CosyVoice3-0.5B-2512](https://huggingface.co/FunAudioLLM/Fun-CosyVoice3-0.5B-2512) |

In [ ]:
from modelscope.pipelines import pipeline
from modelscope.utils.constant import Tasks

def test_cosyvoice3():
    pipe = pipeline(
        task=Tasks.text_to_speech,
        model="FunAudioLLM/Fun-CosyVoice3-0.5B-2512",
        device="cuda:0"
    )

    wavs, timings = [], []
    for text in TEXT_LIST:
        t0 = time.time()
        output = pipe(input=text)
        dt_ms = (time.time() - t0) * 1000
        wav_arr = output["output"]["wave"]
        wavs.append(wav_arr)
        timings.append(SentenceTiming(text=text[:30]+"...", time_ms=round(dt_ms, 1),
                                       audio_len_s=round(len(wav_arr)/24000, 2)))

    combined = np.concatenate(wavs)
    sf.write(str(OUT_DIR / "02_cosyvoice3.wav"), combined, 24000)
    return combined, 24000, timings

wav, sr = bench("Fun-CosyVoice3", test_cosyvoice3)
ipd.Audio(wav, rate=sr)

---
## Model 3: IndexTTS-1.5 — 工业级中文专优

| 属性 | 值 |
|------|-----|
| 参数量 | GPT-2 AR + BigVGAN (模型大小 ~1.3GB) |
| 语言 | 中文 + 英文，原生中英混合 |
| 中文优化 | 拼音纠错，多音字完美处理 |
| 来源 | B站AI团队 (IndexTeam) |
| 权重 | [IndexTeam/IndexTTS-1.5](https://huggingface.co/IndexTeam/IndexTTS-1.5) |
| GGUF | [cstr/indextts-1.5-GGUF](https://huggingface.co/cstr/indextts-1.5-GGUF) (低显存) |

In [ ]:
def test_indextts():
    from indextts.inference import IndexTTS

    model = IndexTTS.from_pretrained("IndexTeam/IndexTTS-1.5", device="cuda")

    wavs, timings = [], []
    for text in TEXT_LIST:
        t0 = time.time()
        audio = model.infer(text=text)
        dt_ms = (time.time() - t0) * 1000
        wav_arr = audio.squeeze()
        wavs.append(wav_arr)
        timings.append(SentenceTiming(text=text[:30]+"...", time_ms=round(dt_ms, 1),
                                       audio_len_s=round(len(wav_arr)/24000, 2)))

    combined = np.concatenate(wavs)
    sf.write(str(OUT_DIR / "03_indextts.wav"), combined, 24000)
    return combined, 24000, timings

wav, sr = bench("IndexTTS-1.5", test_indextts)
ipd.Audio(wav, rate=sr)

---
## Model 4: OmniVoice — 600+语种速度之王

| 属性 | 值 |
|------|-----|
| 参数量 | ~0.8B (Qwen3-0.6B 初始化) |
| 语言 | 646种语言 |
| 中文WER | 0.84% (超ElevenLabs) |
| RTF | 低至 0.025 (40倍实时) |
| 克隆 | 3秒零样本 + 语音设计 (文字描述音色) |
| 协议 | Apache-2.0 (可商用) |
| 权重 | [k2-fsa/OmniVoice](https://huggingface.co/k2-fsa/OmniVoice) |

In [ ]:
from omnivoice import OmniVoice

def test_omnivoice():
    model = OmniVoice.from_pretrained(
        "k2-fsa/OmniVoice",
        device_map="cuda:0",
        dtype=torch.float16
    )

    wavs, timings = [], []
    for text in TEXT_LIST:
        t0 = time.time()
        audio = model.generate(text=text)
        dt_ms = (time.time() - t0) * 1000
        wav_arr = audio.squeeze()
        wavs.append(wav_arr)
        timings.append(SentenceTiming(text=text[:30]+"...", time_ms=round(dt_ms, 1),
                                       audio_len_s=round(len(wav_arr)/24000, 2)))

    combined = np.concatenate(wavs)
    sf.write(str(OUT_DIR / "04_omnivoice.wav"), combined, 24000)
    return combined, 24000, timings

wav, sr = bench("OmniVoice", test_omnivoice)
ipd.Audio(wav, rate=sr)

---
## Model 5: VoxCPM2 — 高保真旗舰克隆

| 属性 | 值 |
|------|-----|
| 参数量 | 2B (MiniCPM-4 backbone) |
| 语言 | 30种语言 + 9种中文方言 |
| 采样率 | 48kHz 录音室级 (AudioVAE V2) |
| 克隆 | Voice Design + Controllable Cloning + Ultimate Cloning |
| VRAM | 约8GB (bf16)，T4可跑 |
| 协议 | Apache-2.0 (可商用) |
| 权重 | [openbmb/VoxCPM2](https://huggingface.co/openbmb/VoxCPM2) |

In [ ]:
from voxcpm import VoxCPM

def test_voxcpm2():
    model = VoxCPM.from_pretrained(
        "openbmb/VoxCPM2",
        load_denoiser=False,
        device_map="cuda:0",
        dtype=torch.float16
    )

    wavs, timings = [], []
    for text in TEXT_LIST:
        t0 = time.time()
        audio = model.generate(text=text)
        dt_ms = (time.time() - t0) * 1000
        wav_arr = audio.squeeze()
        wavs.append(wav_arr)
        timings.append(SentenceTiming(text=text[:30]+"...", time_ms=round(dt_ms, 1),
                                       audio_len_s=round(len(wav_arr)/48000, 2)))

    combined = np.concatenate(wavs)
    sf.write(str(OUT_DIR / "05_voxcpm2.wav"), combined, 48000)
    return combined, 48000, timings

try:
    wav, sr = bench("VoxCPM2", test_voxcpm2)
    ipd.Audio(wav, rate=sr)
except torch.cuda.OutOfMemoryError:
    print("  ❌ OOM! 尝试: Runtime → 断开并删除运行时 → 重新连接 → 仅运行此Cell")
    RESULTS["VoxCPM2"] = TTSResult(model="VoxCPM2", notes="OOM — 重连后重试")

---
## Model 6: S1-Mini — 情感丰富轻量备选

| 属性 | 值 |
|------|-----|
| 参数量 | ~0.5B (从S1-4B蒸馏) |
| 语言 | 14种语言 (中英日法等) |
| 情感 | 50+ 情感和语气 (生气/开心/惊讶/笑声/哭声) |
| WER | 0.011 |
| 协议 | CC-BY-NC-SA-4.0 (非商用) |
| 权重 | [fishaudio/openaudio-s1-mini](https://huggingface.co/fishaudio/openaudio-s1-mini) |

In [ ]:
from fish_speech.inference_engine import TTSInferenceEngine

def test_s1mini():
    engine = TTSInferenceEngine(
        llama_checkpoint_path="fishaudio/openaudio-s1-mini",
        decoder_checkpoint_path="fishaudio/openaudio-s1-mini/firefly-gan-vq-fsq-8x1024-21hz-generator.pth",
        device="cuda",
        compile=False,
        half=True
    )

    wavs, timings = [], []
    for text in TEXT_LIST:
        t0 = time.time()
        audio = engine.tts(text=text, max_new_tokens=1024)
        dt_ms = (time.time() - t0) * 1000
        wavs.append(audio)
        timings.append(SentenceTiming(text=text[:30]+"...", time_ms=round(dt_ms, 1),
                                       audio_len_s=round(len(audio)/44100, 2)))

    combined = np.concatenate(wavs)
    sf.write(str(OUT_DIR / "06_s1mini.wav"), combined, 44100)
    return combined, 44100, timings

wav, sr = bench("S1-Mini", test_s1mini)
ipd.Audio(wav, rate=sr)

---
## 综合评测结果

In [ ]:
import pandas as pd

PARAMS = {
    "ChatTTS": "30M", "Fun-CosyVoice3": "0.5B", "IndexTTS-1.5": "~1.3GB",
    "OmniVoice": "0.8B", "VoxCPM2": "2B", "S1-Mini": "0.5B"
}

print("=" * 80)
print("  TTS SOTA 小模型评测报告 — Colab T4")
print(f"  GPU: {gpu_name.strip()}  |  VRAM: {vram_mb.strip()}MB  |  Date: 2026-05")
print("=" * 80)

# ============================================================
# Summary table
# ============================================================
print("\n---  综合指标总览 ---")
rows = []
for name, r in RESULTS.items():
    rows.append({
        "模型": name,
        "参数量": PARAMS.get(name, "?"),
        "总耗时(ms)": r.time_ms,
        "总耗时(s)": r.time_s,
        "VRAM(GB)": r.vram_gb,
        "RTF": r.rtf,
        "音频(s)": r.audio_len_s,
        "CPU avg%": r.cpu_avg_pct,
        "CPU peak%": r.cpu_peak_pct,
        "GPU avg%": r.gpu_avg_pct,
        "GPU peak%": r.gpu_peak_pct,
        "备注": r.notes
    })

df = pd.DataFrame(rows).sort_values("总耗时(ms)")
print(df.to_markdown(index=False, floatfmt=".1f"))

# ============================================================
# Per-sentence comparison table (all models vs same text)
# ============================================================
print("\n---  逐句推理耗时对比 (ms) — 统一文本 ---")
print(f"  Text[0]: '{TEXT_LIST[0]}'")
print(f"  Text[1]: '{TEXT_LIST[1][:40]}...'")
print(f"  Text[2]: '{TEXT_LIST[2]}'")
print()

sentence_data = []
for name, r in RESULTS.items():
    if not r.sentences:
        continue
    row = {"模型": name}
    for i, st in enumerate(r.sentences):
        row[f"S{i} (ms)"] = st.time_ms
        row[f"S{i} 音频(s)"] = st.audio_len_s
    sentence_data.append(row)

if sentence_data:
    sdf = pd.DataFrame(sentence_data).set_index("模型")
    # Sort by total S0+S1+S2
    sdf["总计(ms)"] = sdf[[c for c in sdf.columns if "ms" in c]].sum(axis=1)
    sdf = sdf.sort_values("总计(ms)")
    print(sdf.to_markdown(floatfmt=".1f"))
else:
    print("  (no per-sentence data yet)")

# ============================================================
# RTF ranking
# ============================================================
print("\n---  速度排名 (RTF 越小越快，<1 = 实时) ---")
valid = {k: v for k, v in RESULTS.items() if v.rtf < 900}
for i, (name, r) in enumerate(sorted(valid.items(), key=lambda x: x[1].rtf)):
    tag = "✅ 实时" if r.rtf < 1 else f"⚠ {r.rtf:.1f}x 实时"
    print(f"  {i+1}. {name}: RTF={r.rtf} ({tag}) | {r.time_ms:.0f}ms | VRAM={r.vram_gb}GB")

# ============================================================
# CPU/GPU utilization comparison
# ============================================================
print("\n---  CPU / GPU 使用率对比 ---")
for name, r in sorted(RESULTS.items(), key=lambda x: x[1].gpu_avg_pct, reverse=True):
    print(f"  {name}: CPU avg {r.cpu_avg_pct:.1f}% / peak {r.cpu_peak_pct:.1f}%  |  GPU avg {r.gpu_avg_pct:.1f}% / peak {r.gpu_peak_pct:.1f}%")

# ============================================================
# T4 VRAM fit
# ============================================================
print("\n---  T4 (15GB) 适配评估 ---")
for name, r in RESULTS.items():
    vram = r.vram_gb
    if isinstance(vram, (int, float)) and vram > 0:
        if vram < 4:    status = "✅ 极轻量"
        elif vram < 8:  status = "✅ 舒适"
        elif vram < 13: status = "✅ 正常"
        elif vram < 15: status = "⚠️ 紧张"
        else:           status = "❌ OOM"
        print(f"  {name}: peak {vram:.1f}GB → {status}")
    elif r.notes:
        print(f"  {name}: {r.notes}")

# ============================================================
# Recommendations
# ============================================================
print("\n---  场景选型建议 ---")
print("  极致轻量 / 实时对话     → ChatTTS (30M, 树莓派可跑)")
print("  中文综合最佳 / 方言场景  → Fun-CosyVoice3 (18方言, 3s克隆)")
print("  中文朗读 / 多音字精准    → IndexTTS-1.5 (拼音纠错)")
print("  超高速多语种 / 商用      → OmniVoice (RTF 0.025, Apache-2.0)")
print("  顶级克隆质量 / 48kHz     → VoxCPM2 (SIM 82.5%, 可商用)")
print("  轻量情感 / 非商用        → S1-Mini (50+情感, CC-BY-NC-SA)")

---

## 附录 A: 权重来源验证

全部6个模型权重经 2026-05 验证，均可公开下载:

| 模型 | HuggingFace | GitHub | pip |
|------|------------|--------|-----|
| ChatTTS | [2noise/ChatTTS](https://huggingface.co/2noise/ChatTTS) | [2noise/ChatTTS](https://github.com/2noise/ChatTTS) | `pip install ChatTTS` |
| Fun-CosyVoice3 | [FunAudioLLM/Fun-CosyVoice3-0.5B-2512](https://huggingface.co/FunAudioLLM/Fun-CosyVoice3-0.5B-2512) | [FunAudioLLM/CosyVoice](https://github.com/FunAudioLLM/CosyVoice) | `pip install cosyvoice` |
| IndexTTS-1.5 | [IndexTeam/IndexTTS-1.5](https://huggingface.co/IndexTeam/IndexTTS-1.5) | [index-tts/index-tts](https://github.com/index-tts/index-tts) | `pip install index-tts` |
| OmniVoice | [k2-fsa/OmniVoice](https://huggingface.co/k2-fsa/OmniVoice) | [k2-fsa/OmniVoice](https://github.com/k2-fsa/OmniVoice) | `pip install omnivoice` |
| VoxCPM2 | [openbmb/VoxCPM2](https://huggingface.co/openbmb/VoxCPM2) | [OpenBMB/VoxCPM](https://github.com/OpenBMB/VoxCPM) | `pip install voxcpm` |
| S1-Mini | [fishaudio/openaudio-s1-mini](https://huggingface.co/fishaudio/openaudio-s1-mini) | [fishaudio/fish-speech](https://github.com/fishaudio/fish-speech) | `pip install fish-speech` |

## 附录 B: Colab T4 运行指南

1. **首次运行:** Cell 1 安装依赖 + 自动下载权重 (~10-20分钟)
2. **逐模型运行:** 每模型独立Cell，互不依赖，可单独测试
3. **OOM处理:** VoxCPM2 (2B) 可能OOM → 断开运行时 → 重新连接 → 仅跑该Cell
4. **音频文件:** `/content/tts_outputs/` 目录，右键下载
5. **免费超时:** ~4-6小时，优先跑 ChatTTS → Fun-CosyVoice3 → OmniVoice